# 12 - POI Grouping

This notebook explores how to group fragmented POIs into broader landmark families and distinct visit entities.

Goals:
- identify likely fragmented landmark families
- detect nearby POIs with similar names
- prepare a grouping candidate table

In [13]:
import re
import unicodedata
import numpy as np
import pandas as pd

from math import radians, sin, cos, sqrt, atan2

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load enriched POIs

In [14]:
pois = pd.read_csv("../data/processed/poi_enriched.csv")
pois.head()

,poi_id,name,name_en,category,category_clean,lat,lon,distance_to_center_km,nearby_count_500m,cluster_id,category_score,landmark_name_score,centrality_score,density_score,importance_score,is_park,is_historic,is_museum,is_attraction,is_religious
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007586,28.975554,0.248384,64,12,0.82,0.55,0.997367,0.941176,0.863004,0,1,0,0,0
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007289,28.975329,0.276907,63,12,0.82,0.55,0.997021,0.926471,0.859224,0,1,0,0,0
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic:castle,historic,41.006397,28.974808,0.361956,63,12,0.82,0.55,0.995990,0.926471,0.858915,0,1,0,0,0
3,311681431,Sağlık Müzesi,NaN,tourism:museum,museum,41.008314,28.975290,0.261290,66,12,0.85,0.35,0.997210,0.970588,0.849310,0,0,1,0,0
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,tourism:museum,museum,41.004295,28.977433,0.441766,55,12,0.85,0.59,0.995023,0.808824,0.844213,0,0,1,0,0


In [15]:
print("Shape:", pois.shape)
print(pois.columns.tolist())

Shape: (2761, 20)
['poi_id', 'name', 'name_en', 'category', 'category_clean', 'lat', 'lon', 'distance_to_center_km', 'nearby_count_500m', 'cluster_id', 'category_score', 'landmark_name_score', 'centrality_score', 'density_score', 'importance_score', 'is_park', 'is_historic', 'is_museum', 'is_attraction', 'is_religious']


In [ ]:
grouping_pois = (
    pois[pois["category_clean"].isin(["museum", "historic", "attraction"])]
    .sort_values("importance_score", ascending=False)
    .head(120)
    .copy()
)

print("Grouping subset shape:", grouping_pois.shape)
grouping_pois[["name", "name_en", "category_clean", "importance_score"]].head(20)

## Name normalization helpers
We create normalized names so we can compare POIs more consistently.

In [16]:
GENERIC_WORDS = {
    "museum", "müzesi", "muzesi",
    "mosque", "cami", "camii",
    "palace", "saray", "sarayı", "sarayi",
    "park", "tower", "kule",
    "square", "meydan", "meydani", "meydanı",
    "cistern", "sarnic", "sarnıcı", "sarnici",
    "church", "kilise",
    "tomb", "türbe", "turbe",
    "mausoleum", "of", "the", "and"
}


def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).casefold()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_meaningful(text):
    tokens = normalize_text(text).split()
    return [tok for tok in tokens if tok not in GENERIC_WORDS and len(tok) >= 3]


def root_name(text):
    tokens = tokenize_meaningful(text)
    return " ".join(tokens)

In [ ]:
grouping_pois["name_norm"] = grouping_pois["name"].apply(normalize_text)
grouping_pois["name_root"] = grouping_pois["name"].apply(root_name)

if "name_en" in grouping_pois.columns:
    grouping_pois["name_en_norm"] = grouping_pois["name_en"].fillna("").apply(normalize_text)
    grouping_pois["name_en_root"] = grouping_pois["name_en"].fillna("").apply(root_name)
else:
    grouping_pois["name_en_norm"] = ""
    grouping_pois["name_en_root"] = ""

grouping_pois[["name", "name_en", "name_root", "name_en_root"]].head(20)

,name,name_en,name_root,name_en_root
0,Lausos Sarayı'nın Kalıntıları,NaN,lausos kal lar,
1,Antiochos Sarayı'nın Kalıntıları,NaN,antiochos kal lar,
2,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,ibrahim pasa,ibrahim pasha
3,Sağlık Müzesi,NaN,sagl,
4,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,buyuk mozaikleri,great mosaic
5,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,sultan ahmet turbesi,sultan ahmed
6,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,ayasofya tarih deneyim,hagia sophia history experience
7,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,turk islam eserleri,turkish islamic arts
8,Halı Müzesi,Carpet Museum,hal,carpet
9,Keçecizâde Fuad Paşa Türbesi,Turbe of Keçecizâde Fuad Pasha,kececizade fuad pasa turbesi,kececizade fuad pasha


## Distance helper
We use geographic distance as another grouping clue.

In [18]:
EARTH_RADIUS_KM = 6371.0088

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return EARTH_RADIUS_KM * c

## Quick root-name frequency check
This helps identify repeated landmark roots.

In [19]:
root_counts = (
    pois["name_root"]
    .value_counts()
    .reset_index()
)

root_counts.columns = ["name_root", "count"]
root_counts[root_counts["name_root"] != ""].head(30)

,name_root,count
0,korugan,83
2,ataturk,26
3,cumhuriyet,18
4,cocuk,15
5,istanbul surlar,14
6,mimar sinan,13
7,yasam vadisi,11
8,caml,9
9,selale,8
10,bar manco,8


## Inspect repeated root names

In [20]:
repeated_roots = root_counts[root_counts["count"] >= 2]["name_root"].tolist()

candidate_repeats = pois[pois["name_root"].isin(repeated_roots)][[
    "poi_id",
    "name",
    "name_en",
    "category_clean",
    "cluster_id",
    "lat",
    "lon",
    "importance_score",
    "name_root",
    "name_en_root"
]].sort_values(["name_root", "importance_score"], ascending=[True, False])

candidate_repeats.head(50)

,poi_id,name,name_en,category_clean,cluster_id,lat,lon,importance_score,name_root,name_en_root
42,1272157516,SARNIÇ CISTERN,NaN,historic,12,41.012085,28.983844,0.773386,,
209,4477143891,Yıldız Sarayı,Yıldız Palace,attraction,10,41.050711,29.011704,0.644443,,
388,5052980822,فندقنا,NaN,attraction,12,41.034156,28.983957,0.593630,,
391,5311889622,صرافی,Beer Corner Rock Pubصرافی,attraction,12,41.035745,28.981708,0.593078,,beer corner rock pub
492,10286582210,Yıldız Parkı,Yildiz Park,attraction,10,41.046874,29.017485,0.573716,,yildiz
739,71898102,Saray Meydanı,NaN,park,17,41.014932,28.928928,0.516983,,
777,6781557685,منزل ايهاب أبو عرمانة,NaN,attraction,5,41.051765,28.837693,0.506402,,
818,5113470962,Kırımlı park,NaN,park,17,41.016571,28.924320,0.488565,,
975,739659671,Mıstık Parkı,NaN,park,12,41.051747,28.993055,0.461507,,
1054,24807026,Fındıklı Parkı,NaN,park,12,41.031473,28.990024,0.454911,,


## Pairwise grouping candidates within the same cluster
We only compare POIs inside the same cluster to keep the search realistic.

In [21]:
def root_overlap_score(row_a, row_b):
    tokens_a = set(tokenize_meaningful(row_a["name"]))
    tokens_b = set(tokenize_meaningful(row_b["name"]))

    if row_a.get("name_en", ""):
        tokens_a.update(tokenize_meaningful(row_a["name_en"]))
    if row_b.get("name_en", ""):
        tokens_b.update(tokenize_meaningful(row_b["name_en"]))

    if not tokens_a or not tokens_b:
        return 0.0

    return len(tokens_a.intersection(tokens_b)) / min(len(tokens_a), len(tokens_b))

In [22]:
pair_candidates = []

cluster_subset = pois.sort_values("importance_score", ascending=False).head(300).copy()

for cluster_id, group in cluster_subset.groupby("cluster_id"):
    group = group.reset_index(drop=True)

    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            row_a = group.iloc[i]
            row_b = group.iloc[j]

            distance_km = haversine_km(
                row_a["lat"], row_a["lon"],
                row_b["lat"], row_b["lon"]
            )

            if distance_km > 0.5:
                continue

            overlap = root_overlap_score(row_a, row_b)

            if overlap == 0:
                continue

            same_category = int(row_a["category_clean"] == row_b["category_clean"])

            pair_candidates.append({
                "poi_id_1": row_a["poi_id"],
                "name_1": row_a["name"],
                "name_en_1": row_a.get("name_en", ""),
                "category_1": row_a["category_clean"],
                "poi_id_2": row_b["poi_id"],
                "name_2": row_b["name"],
                "name_en_2": row_b.get("name_en", ""),
                "category_2": row_b["category_clean"],
                "cluster_id": cluster_id,
                "distance_km": distance_km,
                "root_overlap_score": overlap,
                "same_category": same_category,
                "importance_1": row_a["importance_score"],
                "importance_2": row_b["importance_score"],
            })

pair_candidates_df = pd.DataFrame(pair_candidates)
pair_candidates_df.head(30)

,poi_id_1,name_1,name_en_1,category_1,poi_id_2,name_2,name_en_2,category_2,cluster_id,distance_km,root_overlap_score,same_category,importance_1,importance_2
0,733808433,Aziz Mahmut Hüdayi Türbesi,NaN,historic,1111181701,Cennet Efendi Türbesi,NaN,historic,1,0.027993,0.333333,1,0.615650,0.615548
1,498861861,Çırağan Sarayı,Ciragan Palace,historic,895150392,Çırağan Sarayı Geçidi,NaN,historic,10,0.090528,0.500000,1,0.639858,0.639632
2,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,historic,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,historic,12,0.038034,0.666667,1,0.863004,0.859224
3,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic,527580309,Keçecizâde Fuad Paşa Türbesi,Turbe of Keçecizâde Fuad Pasha,historic,12,0.165908,0.666667,1,0.858915,0.821057
4,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic,3281285899,Mithat Paşa,NaN,historic,12,0.270333,0.500000,1,0.858915,0.720655
5,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic,111002508,Gazi Atik Ali Paşa Camii,NaN,historic,12,0.442675,0.333333,1,0.858915,0.715883
6,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic,499944905,Atik Ali Paşa Medresesi,NaN,historic,12,0.457501,0.333333,1,0.858915,0.694578
7,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic,3374681017,Atik Ali Paşa Medresesi,NaN,historic,12,0.462677,0.333333,1,0.858915,0.687220
8,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,attraction,527580309,Keçecizâde Fuad Paşa Türbesi,Turbe of Keçecizâde Fuad Pasha,historic,12,0.348532,0.250000,0,0.835898,0.821057
9,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,attraction,132277807,Şehzadeler Türbesi,Mausoleum of Princes,historic,12,0.243633,0.333333,0,0.835898,0.807860


## Strong grouping candidates
These are potential same-family or same-entity pairs.

In [23]:
strong_pairs = pair_candidates_df[
    (pair_candidates_df["root_overlap_score"] >= 0.5) &
    (pair_candidates_df["distance_km"] <= 0.3)
].sort_values(
    ["root_overlap_score", "distance_km"],
    ascending=[False, True]
)

strong_pairs.head(50)

,poi_id_1,name_1,name_en_1,category_1,poi_id_2,name_2,name_en_2,category_2,cluster_id,distance_km,root_overlap_score,same_category,importance_1,importance_2
327,5112750831,İvaz Efendi Çeşmesi,NaN,historic,1202955220,İvaz Efendi Çeşmesi,NaN,historic,12,0.000089,1.000000,1,0.625467,0.625467
207,247442912,Masumiyet Müzesi,NaN,museum,3279398393,Masumiyet Müzesi,The Museum of Innocence,museum,12,0.003800,1.000000,1,0.694032,0.694025
278,9909824529,Orhan Kemal Müzesi,NaN,museum,24533505,Orhan Kemal Müzesi,Orhan Kemal Museum,museum,12,0.006800,1.000000,1,0.653598,0.649897
280,1203332143,Ebu Şeybe El-Hudri Türbesi,NaN,historic,5112750839,Ebu Şeybe El-Hudri Türbesi,NaN,attraction,12,0.007073,1.000000,0,0.651571,0.639575
202,499944905,Atik Ali Paşa Medresesi,NaN,historic,3374681017,Atik Ali Paşa Medresesi,NaN,historic,12,0.008571,1.000000,1,0.694578,0.687220
102,1555271,Ayasofya-i Kebir Câmi-i Şerifi,NaN,museum,109862851,Ayasofya-i Kebir Câmi-i Şerifi,Hagia Sophia,attraction,12,0.012460,1.000000,0,0.768184,0.747139
251,1423624920,Kuşluk,NaN,historic,1423624916,Kuşluk Köşkü,NaN,historic,12,0.029827,1.000000,1,0.664194,0.660626
253,225470917,I. Hareket Köşkü,NaN,historic,225470899,II. Hareket Köşkü,NaN,historic,12,0.030789,1.000000,1,0.663823,0.656381
276,530624261,Yavuz Sultan Selim Türbesi,Tomb of Selim the Resolute,historic,7328115,Yavuz Sultan Selim Cami,Yavuz Sultan Selim Mosque,attraction,12,0.049306,1.000000,0,0.653840,0.630342
46,2472730735,Binbirdirek Sarnıcı,Cistern of Philoxenos,museum,214800348,Binbirdirek Sarnıcı,NaN,park,12,0.057191,1.000000,0,0.804169,0.672931


In [24]:
strong_pairs[[
    "name_1", "name_en_1", "category_1",
    "name_2", "name_en_2", "category_2",
    "distance_km", "root_overlap_score", "same_category"
]].head(30)

,name_1,name_en_1,category_1,name_2,name_en_2,category_2,distance_km,root_overlap_score,same_category
327,İvaz Efendi Çeşmesi,NaN,historic,İvaz Efendi Çeşmesi,NaN,historic,0.000089,1.000000,1
207,Masumiyet Müzesi,NaN,museum,Masumiyet Müzesi,The Museum of Innocence,museum,0.003800,1.000000,1
278,Orhan Kemal Müzesi,NaN,museum,Orhan Kemal Müzesi,Orhan Kemal Museum,museum,0.006800,1.000000,1
280,Ebu Şeybe El-Hudri Türbesi,NaN,historic,Ebu Şeybe El-Hudri Türbesi,NaN,attraction,0.007073,1.000000,0
202,Atik Ali Paşa Medresesi,NaN,historic,Atik Ali Paşa Medresesi,NaN,historic,0.008571,1.000000,1
102,Ayasofya-i Kebir Câmi-i Şerifi,NaN,museum,Ayasofya-i Kebir Câmi-i Şerifi,Hagia Sophia,attraction,0.012460,1.000000,0
251,Kuşluk,NaN,historic,Kuşluk Köşkü,NaN,historic,0.029827,1.000000,1
253,I. Hareket Köşkü,NaN,historic,II. Hareket Köşkü,NaN,historic,0.030789,1.000000,1
276,Yavuz Sultan Selim Türbesi,Tomb of Selim the Resolute,historic,Yavuz Sultan Selim Cami,Yavuz Sultan Selim Mosque,attraction,0.049306,1.000000,0
46,Binbirdirek Sarnıcı,Cistern of Philoxenos,museum,Binbirdirek Sarnıcı,NaN,park,0.057191,1.000000,0


## Candidate landmark-family review table
We prepare a manual review file for likely grouped POIs.

In [25]:
group_review = strong_pairs.copy()

group_review["group_decision"] = ""
group_review["group_type"] = ""
group_review["family_name"] = ""
group_review["entity_name"] = ""
group_review["notes"] = ""

group_review.to_csv("../data/processed/poi_grouping_candidates.csv", index=False)

print("Saved: ../data/processed/poi_grouping_candidates.csv")
print("Rows:", len(group_review))
group_review.head(30)

Saved: ../data/processed/poi_grouping_candidates.csv
Rows: 113


,poi_id_1,name_1,name_en_1,category_1,poi_id_2,name_2,name_en_2,category_2,cluster_id,distance_km,root_overlap_score,same_category,importance_1,importance_2,group_decision,group_type,family_name,entity_name,notes
327,5112750831,İvaz Efendi Çeşmesi,NaN,historic,1202955220,İvaz Efendi Çeşmesi,NaN,historic,12,0.000089,1.000000,1,0.625467,0.625467,,,,,
207,247442912,Masumiyet Müzesi,NaN,museum,3279398393,Masumiyet Müzesi,The Museum of Innocence,museum,12,0.003800,1.000000,1,0.694032,0.694025,,,,,
278,9909824529,Orhan Kemal Müzesi,NaN,museum,24533505,Orhan Kemal Müzesi,Orhan Kemal Museum,museum,12,0.006800,1.000000,1,0.653598,0.649897,,,,,
280,1203332143,Ebu Şeybe El-Hudri Türbesi,NaN,historic,5112750839,Ebu Şeybe El-Hudri Türbesi,NaN,attraction,12,0.007073,1.000000,0,0.651571,0.639575,,,,,
202,499944905,Atik Ali Paşa Medresesi,NaN,historic,3374681017,Atik Ali Paşa Medresesi,NaN,historic,12,0.008571,1.000000,1,0.694578,0.687220,,,,,
102,1555271,Ayasofya-i Kebir Câmi-i Şerifi,NaN,museum,109862851,Ayasofya-i Kebir Câmi-i Şerifi,Hagia Sophia,attraction,12,0.012460,1.000000,0,0.768184,0.747139,,,,,
251,1423624920,Kuşluk,NaN,historic,1423624916,Kuşluk Köşkü,NaN,historic,12,0.029827,1.000000,1,0.664194,0.660626,,,,,
253,225470917,I. Hareket Köşkü,NaN,historic,225470899,II. Hareket Köşkü,NaN,historic,12,0.030789,1.000000,1,0.663823,0.656381,,,,,
276,530624261,Yavuz Sultan Selim Türbesi,Tomb of Selim the Resolute,historic,7328115,Yavuz Sultan Selim Cami,Yavuz Sultan Selim Mosque,attraction,12,0.049306,1.000000,0,0.653840,0.630342,,,,,
46,2472730735,Binbirdirek Sarnıcı,Cistern of Philoxenos,museum,214800348,Binbirdirek Sarnıcı,NaN,park,12,0.057191,1.000000,0,0.804169,0.672931,,,,,


## How to label grouping candidates

Suggested labels:

- `group_decision`
  - `yes`
  - `no`

- `group_type`
  - `same_entity`
  - `same_family`
  - `different`

Use:
- `same_entity` for duplicates / same destination
- `same_family` for related but distinct attractions
- `different` for unrelated POIs